**Definitions**

The lattice parameter is written, with three lattice vectors $\alpha$, $\beta$ and $\gamma$ as:
$$
L = \begin{pmatrix}
\alpha_x & \alpha_y & \alpha_z\\
\beta_x & \beta_y & \beta_z\\
\gamma_x & \gamma_y & \gamma_z\\
\end{pmatrix}
$$
and the transformation between primitive cell lattice and unit cell lattice is given by a matrix $P$:
$$L_p = P L_u$$

Atomic position in cartesian coordinate can be calculated from fractional coordinate as:
$$
\begin{pmatrix}
\alpha_x a + \beta_x b + \gamma_x c \\ \alpha_y a + \beta_y b + \gamma_y c \\ \alpha_z a + \beta_z b + \gamma_z c
\end{pmatrix}
= 
\begin{pmatrix}
\alpha_x & \alpha_y & \alpha_z\\
\beta_x & \beta_y & \beta_z\\
\gamma_x & \gamma_y & \gamma_z\\
\end{pmatrix}^T
\begin{pmatrix}
a \\ b \\ c
\end{pmatrix}
$$

Following these definition, we can find:
$$
L_p^T \begin{pmatrix}a_p \\ b_p \\ c_p\end{pmatrix} = 
L_u^T \begin{pmatrix}a_u \\ b_u \\ c_u\end{pmatrix}
$$
which leads to:
$$
\begin{pmatrix}a_p \\ b_p \\ c_p\end{pmatrix} = (L_p^T)^{-1} L_u^T \begin{pmatrix}a_u \\ b_u \\ c_u\end{pmatrix}
$$

From $L_p = P L_u$, we can find:
$$L_u = P^{-1} L_p \quad \quad L_u^T = L_p^T (P^{-1})^T$$

Finally, we find that the coordinate in the primitive cell is given by coordinate in the unit cell as:
$$
\begin{pmatrix}a_p \\ b_p \\ c_p\end{pmatrix} = (P^{-1})^T \begin{pmatrix}a_u \\ b_u \\ c_u\end{pmatrix}
$$

In [1]:
from pymatgen.core.composition import Composition

In [2]:
from itertools import product

In [3]:
import pandas as pd

In [4]:
def calculate_formula(the_site_neighbours, the_composition_str):
    total_composition=''
    for nsites, ele in zip(the_site_neighbours, the_composition_str):
        total_composition += nsites*ele
    this_formula = Composition(total_composition).formula
    return this_formula


In [5]:
def get_cp_composition_bysite(site_neigbours_def: pd.core.frame.DataFrame, this_composition_str:str, ):
    this_result = {}
    for site, site_neighbours in site_neigbours_def.drop(columns='total').iterrows():
        this_result[site] = calculate_formula(site_neighbours, this_composition_str)
    return this_result

In [6]:
from pymatgen.core import Structure as pymatgenStruct
from pymatgen.analysis.local_env import VoronoiNN

In [7]:
from pymatgen.core import Structure as pymatgenStruct
from pymatgen.io.cif import CifWriter

In [8]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


def write_cif_file(lattice, positions, types, filename):
    s = pymatgenStruct(lattice, types, positions)
    CifWriter(s).write_file(filename)


def build_neighors(cell, positions, site_types):

    types = ['Al'] * len(positions)
    s = pymatgenStruct(cell, types, positions)
    a = VoronoiNN()

    solid_angles_threshold = 0.1
    v_list = []
    for isite in range(len(positions)):
        b = a.get_voronoi_polyhedra(s, isite)
        
        neighbors = [
            site[1]['site'].index for site in b.items() if 
            site[1]['solid_angle'] > solid_angles_threshold
        ]
        v_list.append(neighbors)

    representative = {}
    for isite, site in enumerate(site_types):
        if site not in representative.values():
            representative[isite] = site
            
    site_types_short = list(representative.values())
    ntype = len(site_types_short)
    results_matrix = np.zeros((ntype, ntype + 1), dtype=int)
    for isite, type1 in representative.items():
        for isite_neighbor in v_list[isite]:
            type2 = site_types[isite_neighbor]
            i = site_types_short.index(type1)
            j = site_types_short.index(type2)
            results_matrix[i, j] += 1
        results_matrix[i, -1] = len(v_list[isite])

    return pd.DataFrame(results_matrix, columns=site_types_short + ['total'], index=site_types_short)

# **$\sigma$ phase**

$\sigma$ phase has a tetragonal unit cell containing 30 atoms.

In [9]:
a = 10
c = 5
x_4f = 0.4
x_8i1 = 7/15
y_8i1 = 2/15
x_8i2 = 11/15
y_8i2 = 1/15
x_8j = 11/60
z_8j = 1/4

cell = np.array([[a,0,0], [0,a,0], [0,0,c]])

sigma_positions = np.array([
    [0, 0, 0], #a
    [1/2, 1/2, 1/2],
    [x_4f, x_4f, 0], # 4f
    [1-x_4f, 1-x_4f, 0],
    [1/2-x_4f, 1/2+x_4f, 1/2],
    [1/2+x_4f, 1/2-x_4f, 1/2],
    [x_8i1, y_8i1, 0], # 8i1
    [1-x_8i1, 1-y_8i1, 0],
    [1/2-y_8i1, 1/2+x_8i1, 1/2],
    [1/2+y_8i1, 1/2-x_8i1, 1/2],
    [1/2-x_8i1, 1/2+y_8i1, 1/2],
    [1/2+x_8i1, 1/2-y_8i1, 1/2],
    [y_8i1, x_8i1, 0],
    [1-y_8i1, 1-x_8i1, 0],
    [x_8i2, y_8i2, 0], # 8i2
    [1-x_8i2, 1-y_8i2, 0],
    [1/2-y_8i2, -1/2+x_8i2, 1/2],
    [1/2+y_8i2, 3/2-x_8i2, 1/2],
    [3/2-x_8i2, 1/2+y_8i2, 1/2],
    [-1/2+x_8i2, 1/2-y_8i2, 1/2],
    [y_8i2, x_8i2, 0],
    [1-y_8i2, 1-x_8i2, 0],
    [x_8j, x_8j, z_8j], # 8j
    [1-x_8j, 1-x_8j, z_8j],
    [1/2-x_8j, 1/2+x_8j, 1/2+z_8j],
    [1/2+x_8j, 1/2-x_8j, 1/2+z_8j],
    [1/2-x_8j, 1/2+x_8j, 1/2-z_8j],
    [1/2+x_8j, 1/2-x_8j, 1/2-z_8j],
    [x_8j, x_8j, 1-z_8j],
    [1-x_8j, 1-x_8j, 1-z_8j]
])


sigma_neighbours = build_neighors(cell, sigma_positions, ['2$a$']*2 + ['4$f$']*4 + ['8$i_1$']*8 +['8$i_2$']*8 +['8$j$']*8)

In [10]:
sigma_neighbours

,2$a$,4$f$,8$i_1$,8$i_2$,8$j$,total
2$a$,0,4,0,4,4,12
4$f$,2,1,2,4,6,15
8$i_1$,0,1,5,4,4,14
8$i_2$,1,2,4,1,4,12
8$j$,1,3,4,4,2,14


In [11]:
sigma_neighbours.to_latex('sigma_neighbours.tex', float_format='%.0f', header=True, column_format='lcccccc')

the implication of this is that a particular WS has a CP with different compositions, even if the CN is the same. 

In [12]:
composition_neighbour_distribution = {}
for combi  in  product(('A', 'B'), repeat=5):
    this_composition = ''.join(combi)
    composition_neighbour_distribution[this_composition] = get_cp_composition_bysite(sigma_neighbours, this_composition)
composition_neighbour_distribution = pd.DataFrame.from_dict(composition_neighbour_distribution, orient='index')

In [14]:
composition_neighbour_distribution.loc['BAAAA']

2$a$         A12
4$f$      A13 B2
8$i_1$       A14
8$i_2$    A11 B1
8$j$      A13 B1
Name: BAAAA, dtype: object

In [15]:
composition_neighbour_distribution.loc['ABAAA']

2$a$       A8 B4
4$f$      A14 B1
8$i_1$    A13 B1
8$i_2$    A10 B2
8$j$      A11 B3
Name: ABAAA, dtype: object

In [16]:
composition_neighbour_distribution.loc['AABAA']

2$a$         A12
4$f$      A13 B2
8$i_1$     A9 B5
8$i_2$     A8 B4
8$j$      A10 B4
Name: AABAA, dtype: object

In [17]:
composition_neighbour_distribution.loc['AAABA']

2$a$       A8 B4
4$f$      A11 B4
8$i_1$    A10 B4
8$i_2$    A11 B1
8$j$      A10 B4
Name: AAABA, dtype: object

In [13]:
composition_neighbour_distribution.loc['AAAAB']

2$a$       A8 B4
4$f$       A9 B6
8$i_1$    A10 B4
8$i_2$     A8 B4
8$j$      A12 B2
Name: AAAAB, dtype: object

In [12]:
composition_neighbour_distribution.loc['AABAA']

2$a$         A12
4$f$      A13 B2
8$i_1$     A9 B5
8$i_2$     A8 B4
8$j$      A10 B4
Name: AABAA, dtype: object

# **$\chi$ phases**

Body centered cubic lattice with 58 atoms in the unit cell. The cubic lattice with lattice parameter $a$ can be transformed to primitive lattice: 
$$
L_p = \begin{pmatrix}
-a/2& a/2& a/2 \\
a/2& -a/2& a/2 \\
a/2& a/2& -a/2 \\
\end{pmatrix} = \begin{pmatrix} -1/2 & 1/2 & 1/2 \\ 1/2 & -1/2 & 1/2 \\ 1/2 & 1/2 & -1/2 \end{pmatrix} L_u
$$

so that, we have the relaionship between lattice coordinates:
$$\begin{pmatrix}a_p \\ b_p \\ c_p\end{pmatrix} = (P^{-1})^T \begin{pmatrix}a_u \\ b_u \\ c_u\end{pmatrix} = \begin{pmatrix}
0 & 1 & 1 \\
1 & 0 & 1 \\
1 & 1 & 0 \\
\end{pmatrix}
\begin{pmatrix}a_u \\ b_u \\ c_u\end{pmatrix} = \begin{pmatrix}b_u + c_u \\ a_u + c_u \\ a_u + b_u\end{pmatrix}
$$

The fractional translation in body centered cubic lattice $(1/2, 1/2, 1/2)$ is equivalent to translation $(1, 1, 1)$ in the primitive cell lattice. So, it is not necessary to consider the equivalent coordinates with fractional translation.

In [13]:
a = 10.0
x_c = 0.317
x_g1 = 0.356
z_g1 = 0.042
x_g2 = 0.089
z_g2 = 0.278

cell = np.array([
    [-0.5, 0.5, 0.5],
    [0.5, -0.5, 0.5],
    [0.5, 0.5, -0.5],
]) * a

chi_positions = np.array([
    [0, 0, 0], # 2a, CN16
    [2*x_c, 2*x_c, 2*x_c], # 8c, CN16
    [1-2*x_c, 0, 0],
    [0, 1-2*x_c, 0],
    [0, 0, 1-2*x_c],
    [  x_g1+z_g1,   x_g1+z_g1,      2*x_g1], # 24g1, CN13
    [  z_g1-x_g1,   z_g1-x_g1,    1-2*x_g1],
    [  x_g1-z_g1, 1-x_g1-z_g1,           0],
    [1-x_g1-z_g1,   x_g1-z_g1,           0],
    [     2*x_g1,   z_g1+x_g1,   z_g1+x_g1],
    [   1-2*x_g1,   z_g1-x_g1,   z_g1-x_g1],
    [          0,   x_g1-z_g1, 1-x_g1-z_g1],
    [          0, 1-x_g1-z_g1,   x_g1-z_g1],
    [  x_g1+z_g1,      2*x_g1,   x_g1+z_g1],
    [  z_g1-x_g1,    1-2*x_g1,   z_g1-x_g1],
    [1-z_g1-x_g1,           0,   x_g1-z_g1],
    [  x_g1-z_g1,           0, 1-z_g1-x_g1],
    [  x_g2+z_g2,   x_g2+z_g2,      2*x_g2], # 24g2, CN12
    [  z_g2-x_g2,   z_g2-x_g2,    1-2*x_g2],
    [  x_g2-z_g2, 1-x_g2-z_g2,           0],
    [1-x_g2-z_g2,   x_g2-z_g2,           0],
    [     2*x_g2,   z_g2+x_g2,   z_g2+x_g2],
    [   1-2*x_g2,   z_g2-x_g2,   z_g2-x_g2],
    [          0,   x_g2-z_g2, 1-x_g2-z_g2],
    [          0, 1-x_g2-z_g2,   x_g2-z_g2],
    [  x_g2+z_g2,      2*x_g2,   x_g2+z_g2],
    [  z_g2-x_g2,    1-2*x_g2,   z_g2-x_g2],
    [1-z_g2-x_g2,           0,   x_g2-z_g2],
    [  x_g2-z_g2,           0, 1-z_g2-x_g2],
])
chi_positions -= np.floor(chi_positions)

chi_neighbours = build_neighors(cell, chi_positions, ['2$a$']*1 + ['8$c$']*4 + ['24$g_1$']*12 +['24$g_2$']*12)

In [14]:
chi_neighbours.to_latex('chi_neighbours.tex', float_format='%.0f', header=True, column_format='lccccc')

**MgCu2 cubic Laves Structure**

In this laves structure, there are two non-equivalent sites in a cubic FCC lattice. Typically, A atoms are larger and occupy 8a site with CN16. The smaller B atoms occupy the 16d site. There is no adjustable internal parameters.

To transfer the coordinates in the unit cell to a primitive cell:
$$
L_p = \begin{pmatrix}
0 & a/2& a/2 \\
a/2& 0& a/2 \\
a/2& a/2& 0 \\
\end{pmatrix} = \begin{pmatrix} 0 & 1/2 & 1/2 \\ 1/2 & 0 & 1/2 \\ 1/2 & 1/2 & 0 \end{pmatrix} L_u
$$

$$\begin{pmatrix}a_p \\ b_p \\ c_p\end{pmatrix} = (P^{-1})^T \begin{pmatrix}a_u \\ b_u \\ c_u\end{pmatrix} = \begin{pmatrix}
-1 & 1 & 1 \\
1 & -1 & 1 \\
1 & 1 & -1 \\
\end{pmatrix}
\begin{pmatrix}a_u \\ b_u \\ c_u\end{pmatrix} = \begin{pmatrix}b_u + c_u - a_u \\ a_u + c_u - b_u\\ a_u + b_u - c_u\end{pmatrix}
$$

In [15]:
a = 10
cell = np.array([[0, a/2, a/2], [a/2, 0, a/2], [a/2, a/2, 0]])
mgcu2_positions = np.array([ 
       [0.   , 0.   , 0.   ], # A 
       [0.   , 0.5  , 0.5  ],
       [0.5  , 0.   , 0.5  ],
       [0.5  , 0.5  , 0.   ],
       [0.25 , 0.25 , 0.25 ],
       [0.25 , 0.75 , 0.75 ],
       [0.75 , 0.25 , 0.75 ],
       [0.75 , 0.75 , 0.25 ],
       [0.625, 0.625, 0.625],
       [0.625, 0.125, 0.125],
       [0.125, 0.625, 0.125],
       [0.125, 0.125, 0.625],
       [0.625, 0.875, 0.875],
       [0.625, 0.375, 0.375],
       [0.125, 0.875, 0.375],
       [0.125, 0.375, 0.875],
       [0.875, 0.625, 0.875],
       [0.875, 0.125, 0.375],
       [0.375, 0.625, 0.375],
       [0.375, 0.125, 0.875],
       [0.875, 0.875, 0.625],
       [0.875, 0.375, 0.125],
       [0.375, 0.875, 0.125],
       [0.375, 0.375, 0.625]
])
mgcu2_pcell_positions = np.array([
    [0, 0, 0],
    [1/4, 1/4, 1/4],
    [5/8, 5/8, 5/8],
    [1/8, 5/8, 5/8],
    [5/8, 1/8, 5/8],
    [5/8, 5/8, 1/8]
])

c15_neighbours = build_neighors(cell, mgcu2_pcell_positions, ['8$a$']*2 + ['16$d$']*4)

In [16]:
c15_neighbours

,8$a$,16$d$,total
8$a$,4,12,16
16$d$,6,6,12


In [17]:
c15_neighbours.to_latex('c15_neighbours.tex', float_format='%.0f', header=True, column_format='lcccc')

# **MgZn2 Laves Structure** (C14)

We have a hexagonal unitcell with 12 atoms per cell. The lattice vector of the hexagonal cell is:
$$
L = \begin{pmatrix} a/2 & -\sqrt{3} a / 2 & 0 \\ a/2 & \sqrt{3} a / 2 & 0 \\ 0 & 0 & c\end{pmatrix}
$$
with a $c/a$ ratio approximately 1.63. The larger A atoms are expected to occupy the 4f sites while the smaller B atoms are expected to occupy the 2a and 6h sites. The space group is 194 ($P6_3/mmc$)

In [18]:
a = 10
c = 16
z_4f = 1/16
x_6h = -1/6

cell = np.array([
    [0.5 * a, - np.sqrt(3)/2 * a, 0],
    [0.5 * a,   np.sqrt(3)/2 * a, 0],
    [ 0, 0, c]
])
mgzn2_positions = np.array([
    [1/3, 2/3, z_4f], # 4f
    [2/3, 1/3, -z_4f],
    [2/3, 1/3, 1/2+z_4f],
    [1/3, 2/3, 1/2-z_4f],
    [0, 0, 0], # 2a
    [0, 0, 1/2],
    [x_6h, 2 * x_6h, 1/4], # 6h
    [-2 * x_6h, -x_6h, 1/4],
    [x_6h, -x_6h, 1/4],
    [- x_6h, - 2*x_6h, 3/4],
    [2 * x_6h, x_6h, 3/4],
    [- x_6h, x_6h, 3/4]
])

C14_neighbours = build_neighors(cell, mgzn2_positions, ['4$f$']*4 + ['2$a$']*2 +['6$h$']*6)

In [19]:
C14_neighbours.to_latex('C14_neighbours.tex', float_format='%.0f', header=True, column_format='|l|c|c|c|c|')

# **MgNi2 Laves structure** (C36)

For this, we again have a hexagonal cell with space group same as in the MgZn2 structure. A atoms are supposed to occupy $4e$ and $4f$ sites while the B atoms should occupy $6g$, $6h$ and $4f$ sites.

In [20]:
a = 6
c = 6 * 3.24
z_4e = 3/32
z_4f1 = 27/32
x_6h = 1/6
z_4f2 = 1/8
cell = np.array([
    [0.5 * a, - np.sqrt(3)/2 * a, 0],
    [0.5 * a,   np.sqrt(3)/2 * a, 0],
    [ 0, 0, c]
])

mgni2_positions = np.array([
    [0, 0, z_4e], # 4e
    [0, 0, -z_4e],
    [0, 0, 0.5 + z_4e],
    [0, 0, 0.5 - z_4e], # 4f
    [1/3, 2/3, z_4f1],
    [2/3, 1/3, 0.5 + z_4f1],
    [2/3, 1/3, - z_4f1],
    [1/3, 2/3, 0.5-z_4f1],
    [1/2, 0, 0], # 6g
    [0, 1/2, 0],
    [1/2, 1/2, 0],
    [1/2, 0, 1/2],
    [0, 1/2, 1/2],
    [1/2, 1/2, 1/2],
    [x_6h, 2 * x_6h, 1/4], # 6h
    [-2 * x_6h, -x_6h, 1/4],
    [x_6h, -x_6h, 1/4],
    [- x_6h, - 2*x_6h, 3/4],
    [2 * x_6h, x_6h, 3/4],
    [- x_6h, x_6h, 3/4],
    [1/3, 2/3, z_4f2], # 4f
    [2/3, 1/3, 0.5 + z_4f2],
    [2/3, 1/3, - z_4f2],
    [1/3, 2/3, 0.5-z_4f2],
])

c36_neighbours = build_neighors(cell, mgni2_positions, ['4$e$']*4 + ['4$f_1$']*4 + ['6$g$']*6 +['6$h$']*6 +['4$f_2$']*4)

In [21]:
c36_neighbours

,4$e$,4$f_1$,6$g$,6$h$,4$f_2$,total
4$e$,1,3,6,3,3,16
4$f_1$,3,1,3,6,3,16
6$g$,4,2,4,0,2,12
6$h$,2,4,0,4,2,12
4$f_2$,3,3,3,3,0,12


In [22]:
c36_neighbours.to_latex('C36_neighbours.tex', float_format='%.0f', header=True, column_format='|l|c|c|c|c|c|c|')

# **A15 ($\beta$-W) structure**

It has a cubic unit cell in the space group $Pm\overline{3}n$ (223) and often in the stoichiometric composition of A3B. The A atoms typically occupy the 6c position and the B atoms typically occupy the 2a position. There is no adjustable parameters in the internal coordinates.

In [23]:
a = 10
cell = np.eye(3) * a

a15_positions = np.array([
    [0, 0, 0], # 2a
    [1/2, 1/2, 1/2],
    [1/4, 0, 1/2], #6c
    [1/2, 1/4, 0],
    [0, 1/2, 1/4],
    [3/4, 0, 1/2],
    [1/2, 3/4, 0],
    [0, 1/2, 3/4]
])

a15_neighbours = build_neighors(cell, a15_positions, ['2$a$']*2 + ['6$c$']*6)

In [24]:
a15_neighbours

,2$a$,6$c$,total
2$a$,0,12,12
6$c$,4,10,14


In [25]:
a15_neighbours.to_latex('a15_neighbours.tex', float_format='%.0f', header=True, column_format='|l|c|c|c|')

# **P structure**

P structure has primitive orthorhombic cell with 56 atoms in total. There are 12 different crystallographic sites (10 $4c$ and 2 $8d$). There are different settings for the orthorhombic cell. Here I follow the standard setting ($Pnma$). This can be obtained from the $Pbnm$ setting usually used by permutating the axis.

In this setting, the $4c$ is given by:
$$
(x,1/4,z), (\bar{x}+1/2,3/4,z+1/2), (\bar{x},3/4,\bar{z}), (x+1/2,1/4,\bar{z}+1/2)
$$
and the $8d$ site is given by:
$$
(x,y,z), (\bar{x}+1/2, \bar{y}, z+1/2), (\bar{x}, y+1/2, \bar{z}), (x+1/2, \bar{y}+1/2, \bar{z}+1/2) \\
(\bar{x},\bar{y},\bar{z}), (x+1/2, y, \bar{z}+1/2), (x, \bar{y}+1/2, z), (\bar{x}+1/2, y+1/2, z+1/2)
$$

In the case of $\mathrm{Cr_{18}Mo_{42}Ni_{40}}$ structure with the $Pnma$ setting, we have:
$$
a = 16.983;\quad b=4.752 \approx 0.28 a; \quad c=9.07\approx0.534 a
$$

In [ ]:
def get_4c(x, z):
    return np.array([[x, 1/4, z], [1/2-x, 3/4, 1/2+z], [1-x, 3/4, 1-z], [1/2+x, 1/4, 1/2-z]])

def get_8d(x,y,z):
    return np.array([
        [x, y, z], [1/2-x, 1-y, 1/2+z], [1-x, y+1/2, 1-z], [1/2+x, 1/2-y, 1/2-z],
        [1-x,1-y,1-z], [1/2+x, y, 1/2-z], [x, 1/2-y, z], [1/2-x, 1/2+y, 1/2+z]
    ])


In [ ]:


p_positions = np.vstack([
    get_4c(0.1134, 0.0737),
    get_4c(0.2547, 0.1363),
    get_4c(0.1578, 0.3257),
    get_4c(0.1819, 0.6058),
    get_4c(0.3553, 0.6650),
    get_4c(0.4536, 0.4746),
    get_4c(0.4047, 0.1988),
    get_4c(0.0780, 0.8152),
    get_4c(0.3650, 0.9383),
    get_4c(0.0355, 0.5202),
    get_8d(0.5375, 0.9986, 0.2504),
    get_8d(0.2883, 0.0008, 0.3868)
])
cell = np.array([[1, 0, 0], [0, 0.28, 0], [0, 0, 0.545]]) * 16.983

P_neighbours = build_neighors(cell, p_positions, ['4$c_1$']*4 + ['4$c_2$']*4 + ['4$c_3$']*4 + ['4$c_4$']*4 + ['4$c_5$']*4 + ['4$c_6$']*4 + ['4$c_7$']*4 + ['4$c_8$']*4 + ['4$c_9$']*4 + ['4$c_{10}$']*4 + ['8$d_1$']*8 + ['8$d_2$']*8)

In [27]:
P_neighbours.to_latex('P_neighbours.tex', float_format='%.0f', header=True, column_format='|l|'+12*'c|')

In [28]:
P_neighbours

,4$c_1$,4$c_2$,4$c_3$,4$c_4$,4$c_5$,4$c_6$,4$c_7$,4$c_8$,4$c_9$,4$c_{10}$,8$d_1$,8$d_2$,total
4$c_1$,0,1,1,0,2,3,0,1,0,0,2,2,12
4$c_2$,1,0,1,2,2,0,1,0,1,0,0,4,12
4$c_3$,1,1,0,1,2,0,0,0,2,1,2,2,12
4$c_4$,0,2,1,0,1,0,2,1,2,1,0,4,14
4$c_5$,2,2,2,1,0,1,0,0,1,0,2,4,15
4$c_6$,3,0,0,0,1,2,1,3,0,0,4,2,16
4$c_7$,0,1,0,2,0,1,0,2,1,3,2,2,14
4$c_8$,1,0,0,1,0,3,2,0,0,1,2,2,12
4$c_9$,0,1,2,2,1,0,1,0,0,3,2,2,14
4$c_{10}$,0,0,1,1,0,0,3,1,3,2,4,0,15


# **$\delta$- phase structure**

The $\delta$ phase has a orthorbombic structure with space grup 19 ($P2_12_12_1$) with 56 atoms in the primitive cell. All of them is designated as $4a$. It is observed with composition $Mo-Ni$ binaries. The $4a$ wyckoff site give the following positions:

$$
(x, y, z), (1/2-x, 1-y, z+1/2), (1-x, 1/2+y, 1/2-z), (x+1/2, 1/2-y, 1-z)
$$

The cell parameter in the case of MoNi composition is:
$$
a \approx b = 9.108;\quad c = 8.852 \approx 0.972a
$$

In [29]:
def get_4a(x, y, z):
    return np.array([[x, y, z], [1/2-x, 1-y, 1/2+z], [1-x, 1/2+y, 1/2-z], [1/2+x, 1/2-y, 1-z]])

delta_positions = np.vstack([
    get_4a(0.1763, 0.4832, 0.6425),
    get_4a(0.0338, 0.3398, 0.1807),
    get_4a(0.2289, 0.2865, 0.4098),
    get_4a(0.4519, 0.1153, 0.5322),
    get_4a(0.4424, 0.3662, 0.5971),
    get_4a(0.3882, 0.0523, 0.2748),
    get_4a(0.1337, 0.0707, 0.2157),
    get_4a(0.3768, 0.4358, 0.8567),
    get_4a(0.0680, 0.1442, 0.9529),
    get_4a(0.2648, 0.1993, 0.7486),
    get_4a(0.3186, 0.2464, 0.0740),
    get_4a(0.0029, 0.1969, 0.6767),
    get_4a(0.1885, 0.0157, 0.4960),
    get_4a(0.1031, 0.4192, 0.9133)
])

cell = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 0.972]]) * 9.1

delta_neighbours = build_neighors(cell, delta_positions, ['4$c_1$']*4 + ['4$c_2$']*4 + ['4$c_3$']*4 + ['4$c_4$']*4 + ['4$c_5$']*4 + ['4$c_6$']*4 + ['4$c_7$']*4 + ['4$c_8$']*4 + ['4$c_9$']*4 + ['4$c_{10}$']*4 + ['4$c_{11}$']*4 + ['4$c_{12}$']*4 + ['4$c_{13}$']*4 + ['4$c_{14}$']*4)

In [30]:
delta_neighbours.to_latex('delta_neighbours.tex', float_format='%.0f', header=True, column_format='|l|'+14*'c|')

# **R-phase**

$R$ phase is rhombohedral with 53 atoms per cell. The space group is 148 ($R\bar{3}$) with a unit hexagonal cell containing 159 atoms. The transformation from hexagonal cell to rhombohedral cell is given by:

$$
L_p = \begin{pmatrix}
    0 & \sqrt{3}a/3 & c/3\\
    a/2 & -\sqrt{3}a/6 & c/3\\
    -a/2 & -\sqrt{3}a/6 & c/3
\end{pmatrix}; \quad
L_h = \begin{pmatrix} a/2 & -\sqrt{3} a / 2 & 0 \\ a/2 & \sqrt{3} a / 2 & 0 \\ 0 & 0 & c\end{pmatrix}
$$
with cell transformation:
$$
L_p = \begin{pmatrix} -1/3 & 1/3 & 1/3\\2/3 & 1/3 & 1/3\\-1/3 & -2/3 & 1/3\end{pmatrix} L_h
$$

The internal coordinate transform as:

$$\begin{pmatrix}a_p \\ b_p \\ c_p\end{pmatrix} = (P^{-1})^T \begin{pmatrix}a_u \\ b_u \\ c_u\end{pmatrix} = \begin{pmatrix}
-1 & 1 & 1 \\
 1 & 0 & 1 \\
0 & -1 & 1 \\
\end{pmatrix}
\begin{pmatrix}a_u \\ b_u \\ c_u\end{pmatrix} = \begin{pmatrix}b_u + c_u - a_u \\ a_u + c_u\\ c_u - b_u\end{pmatrix}
$$

There are 11 crystallographically different sites: 1 with $3b$, 2 with $6c$, 8 with $18f$. The $3b$ position is $(0,0,1/2)$ in the hexagonal cell which is transformed to $(1/2,1/2,1/2)$ in the rhomobedral cell.

The cell parameter in the hexagonal lattice is:

$$
a = 10.903; \quad c = 19.342 \approx 1.774 a
$$

In [31]:
def get_18f(x, y, z):
    return np.einsum('ij,kj->ki', 
        np.array([[-1, 1, 1],[1, 0, 1], [0, -1, 1]]), 
        np.array([[x, y, z], [-y, x-y, z], [y-x, -x, z], [-x, -y, -z], [y, y-x, -z], [x-y, x, -z]])
    )
def get_6c(z):
    return np.array([[z, z, z], [1-z, 1-z, 1-z]])

r_positions = np.vstack([
    [1/2, 1/2, 1/2],
    get_6c(0.3044),
    get_6c(0.0735),
    get_18f(0.05090,    0.27900,    0.10000),
    get_18f(0.02120,    0.13930,    0.19620),
    get_18f(0.22500,    0.19690,    0.26850),
    get_18f(0.17590,    0.12650,    0.36960),
    get_18f(0.11320,    0.26870,    0.46520),
    get_18f(0.03300,    0.25790,    0.31830),
    get_18f(0.15960,    0.24700,    0.00200),
    get_18f(0.26710,    0.22180,    0.12220)
])
cell = np.array([
    [0, np.sqrt(3)/3, 1.774/3],
    [1/2, -np.sqrt(3)/6, 1.774/3],
    [-1/2, -np.sqrt(3)/6, 1.774/3]
]) * 10.903

R_neighbours = build_neighors(cell, r_positions, ['3$b$']*1 + ['6$c_1$']*2 + ['6$c_2$']*3 + ['18$f_1$']*6 + ['18$f_2$']*6 + ['18$f_3$']*6 + ['18$f_4$']*6 + ['18$f_5$']*6  + ['18$f_6$']*6 + ['18$f_7$']*6 + ['18$f_8$']*6)

In [32]:
R_neighbours.to_latex('R_neighbours.tex', float_format='%.0f', header=True, column_format='|l|'+11*'c|')

# **M Phase**

M phase has a space group Pnam according to Shinha, but this space group is not present in the Bilbao crystallographic the space group is [Pnam](https://cryst.ehu.es/cgi-bin/cryst/programs/nph-wp-list). Also according to Shinha paper it is also $D_{2h}^{16}$ which is also in accordance with the same space group in [Wikipedia](https://en.wikipedia.org/wiki/List_of_space_groups#List_of_orthorhombic)

In [73]:
def get_4c(x, z):
    return [
            [x, 0.25, z],
            [-x+0.5, 0.75, z+0.5],
            [1-x, 0.75, 1-z],
            [x+0.5, 0.25, -z+0.5]
            ]

In [74]:
def get_8d(x, y, z):
    return np.array([
        [x, y, z],
        [-x+0.5, -y, z+0.5],
        [-x, y+0.5, -z],
        [x+0.5, -y+0.5, -z+0.5],
        [-x, -y, -z],
        [x+0.5, y, z+0.5],
        [x, -y+0.5, z],
        [-x+0.5, y+0.5, -z+0.5]
    ])

In [37]:
m_positions = np.vstack(
    [
    get_4c(0.0593, 0.1494),
    get_4c(0.2996, 0.3984),
    get_4c(0.5242, 0.5410),
    get_4c(0.6164, 0.7068),
    get_4c(0.0144, 0.4482),
    get_4c(0.8388, 0.2938),
    get_4c(0.0714, 0.6225),
    get_4c(0.3255, 0.6697),
    get_4c(0.8168, 0.5778), 
    get_8d(0.1118, 0.2951, 0.0048),
    get_8d(0.2550, 0.5450, -0.0031)
    ]
    )

In [70]:
m_positions

array([[ 0.0593,  0.25  ,  0.1494],
       [ 0.4407,  0.75  ,  0.6494],
       [ 0.9407,  0.75  ,  0.8506],
       [ 0.5593,  0.25  ,  0.3506],
       [ 0.2996,  0.25  ,  0.3984],
       [ 0.2004,  0.75  ,  0.8984],
       [ 0.7004,  0.75  ,  0.6016],
       [ 0.7996,  0.25  ,  0.1016],
       [ 0.5242,  0.25  ,  0.541 ],
       [-0.0242,  0.75  ,  1.041 ],
       [ 0.4758,  0.75  ,  0.459 ],
       [ 1.0242,  0.25  , -0.041 ],
       [ 0.6164,  0.25  ,  0.7068],
       [-0.1164,  0.75  ,  1.2068],
       [ 0.3836,  0.75  ,  0.2932],
       [ 1.1164,  0.25  , -0.2068],
       [ 0.0144,  0.25  ,  0.4482],
       [ 0.4856,  0.75  ,  0.9482],
       [ 0.9856,  0.75  ,  0.5518],
       [ 0.5144,  0.25  ,  0.0518],
       [ 0.8388,  0.25  ,  0.2938],
       [-0.3388,  0.75  ,  0.7938],
       [ 0.1612,  0.75  ,  0.7062],
       [ 1.3388,  0.25  ,  0.2062],
       [ 0.0714,  0.25  ,  0.6225],
       [ 0.4286,  0.75  ,  1.1225],
       [ 0.9286,  0.75  ,  0.3775],
       [ 0.5714,  0.25  , -0

In [54]:
m_cell = np.array([[9.303, 0, 0], [0, 16.266, 0], [0, 0, 4.933]])

In [52]:
m_site_types =   ['4$c_1$']*4 + ['4$c_2$']*4 + ['4$c_3$']*4 + ['4$c_4$']*4 + ['4$c_5$']*4 + ['4$c_6$']*4 + ['4$c_7$']*4 + ['4$c_8$']*4 + ['4$c_9$']*4+['8$d_1$']*8 + ['8$d_2$']*8

In [58]:
s = pymatgenStruct(m_cell, ['Al']*len(m_site_types), m_positions)

In [57]:
a = VoronoiNN()

In [65]:
p = a.get_voronoi_polyhedra(s, 2)

In [66]:
len(p)

7

In [67]:
m_neighbours = build_neighors(
    mcell, m_positions, m_site_types
)

In [69]:
m_neighbours

,4$c_1$,4$c_2$,4$c_3$,4$c_4$,4$c_5$,4$c_6$,4$c_7$,4$c_8$,4$c_9$,8$d_1$,8$d_2$,total
4$c_1$,0,1,1,0,1,1,0,0,0,2,0,6
4$c_2$,1,0,1,0,1,1,1,1,0,2,2,10
4$c_3$,1,1,0,1,0,0,1,1,0,2,0,7
4$c_4$,0,0,1,0,0,0,1,1,1,2,0,6
4$c_5$,1,1,0,0,0,1,1,0,1,2,2,9
4$c_6$,1,1,0,0,1,0,0,0,1,2,2,8
4$c_7$,0,1,1,1,1,0,0,2,1,2,2,11
4$c_8$,0,1,1,1,0,0,2,0,1,0,2,8
4$c_9$,0,0,0,1,1,1,1,1,0,2,2,9
8$d_1$,1,1,1,1,1,1,1,0,1,1,4,13


# **$\mu$ phase**

$\mu$ phase has a rhombohedral unit cell. Using the notation for the $R$ phase, we have:
$$
a = 4.756;\quad c = 25.83887 \approx 5.433 a
$$

There are 5 non-equivalent position in this rhombohedral cell with one $1a$ site, one $6h$ site and three $2c$ sites. In total, there are 13 atoms in the cell.

In [33]:
x_6h = 0.09
z_6h = 0.59
x_2c1 = 0.167
x_2c2 = 0.346
x_2c3 = 0.448

mu_positions = np.array([
    [0, 0, 0], # 1a
    [x_6h, x_6h, z_6h], # 6h
    [x_6h, z_6h, x_6h],
    [z_6h, x_6h, x_6h],
    [1-x_6h, 1-x_6h, 1-z_6h],
    [1-x_6h, 1-z_6h, 1-x_6h],
    [1-z_6h, 1-x_6h, 1-x_6h],
    [x_2c1, x_2c1, x_2c1], #2c1
    [1-x_2c1, 1-x_2c1, 1-x_2c1],
    [x_2c2, x_2c2, x_2c2], #2c2
    [1-x_2c2, 1-x_2c2, 1-x_2c2],
    [x_2c3, x_2c3, x_2c3], #2c3
    [1-x_2c3, 1-x_2c3, 1-x_2c3], 
])
cell = np.array([
    [0, np.sqrt(3)/3, 5.433/3],
    [1/2, -np.sqrt(3)/6, 5.433/3],
    [-1/2, -np.sqrt(3)/6, 5.433/3]
]) * 4.756

mu_neighbours = build_neighors(cell, mu_positions, ['1$a$']*1 + ['6$h$']*6 + ['2$c_1$']*2 + ['2$c_2$']*2 + ['2$c_2$']*2)

In [34]:
mu_neighbours.to_latex('mu_neighbours.tex', float_format='%.0f', header=True, column_format='|l|'+5*'c|')

In [35]:
mu_neighbours

,1$a$,6$h$,2$c_1$,2$c_2$,total
1$a$,0,6,0,6,12
6$h$,1,4,2,5,12
2$c_1$,0,6,3,6,15
2$c_2$,3,9,0,4,16
